# 🚀 nanochat GPU Training Pipeline on Kaggle

This notebook runs the complete end-to-end LLM lifecycle from [Karpathy's nanochat](https://github.com/karpathy/nanochat) on Kaggle Cloud GPUs (T4 / P100 / A100):
1. **Environment Setup & GPU Verification**
2. **Dataset Shard Download & Tokenizer Training**
3. **Pretraining (Base Model)**
4. **Evaluation (CORE metric / BPB)**
5. **Supervised Fine-Tuning (SFT)**
6. **Inference & Interactive Chat**

In [ ]:
# 1. Verify GPU Availability
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Count: {torch.cuda.device_count()}")
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Clone or pull repo and install dependencies
import os
if not os.path.exists("nanochat"):
    !git clone https://github.com/NB7551498/nanoGPT-2.0.git nanochat
%cd nanochat
!pip install -q tiktoken datasets wandb tqdm rustbpe
!pip install -e .

In [ ]:
# 3. Download Data Shards & Train Tokenizer
!python -m nanochat.dataset -n 8
!python -m scripts.tok_train
!python -m scripts.tok_eval

In [ ]:
# 4. Pretrain Base Model
# Note: depth=8 or 12 provides fast training on Kaggle T4/P100 GPUs
import os
n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 1
cmd = f"torchrun --standalone --nproc_per_node={n_gpus} -m scripts.base_train -- --depth=8 --device-batch-size=8 --run=kaggle_run"
print(f"Running command: {cmd}")
!{cmd}

In [ ]:
# 5. Base Model Evaluation
n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 1
cmd_eval = f"torchrun --standalone --nproc_per_node={n_gpus} -m scripts.base_eval -- --device-batch-size=8"
!{cmd_eval}

In [ ]:
# 6. Supervised Fine-Tuning (SFT)
n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 1
!torchrun --standalone --nproc_per_node={n_gpus} -m scripts.chat_sft -- --run=kaggle_sft
!torchrun --standalone --nproc_per_node={n_gpus} -m scripts.chat_eval -- -i sft

In [ ]:
# 7. Test Interactive Chat Inference
!python -m scripts.chat_cli -p "Explain why the sky is blue in 2 sentences."